# Tutorial: Futures Data Walkthrough

Audience:
- new `pysystemtrade` users who want a guided tour of the futures-data pipeline.

Prerequisites:
- A local checkout of this repository.
- Basic familiarity with Python, pandas, and Jupyter notebooks.

Learning goals:
- Map `docs/data.md` onto the real data files and Python objects in this repo.
- Inspect representative futures data assets without needing MongoDB or Interactive Brokers.
- Identify where the sim and production interfaces begin.


## Orientation

This notebook is a guided, runnable companion to `docs/data.md`. It follows the document's four-part structure, but it uses small live inspection cells so you can see the actual files and modules that back the futures-data workflow.


In [ ]:
from __future__ import annotations

from pathlib import Path
import re

import pandas as pd


def find_repo_root(start: Path | None = None) -> Path:
    root = (start or Path.cwd()).resolve()
    for candidate in (root, *root.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "docs" / "data.md").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root from the notebook")


REPO_ROOT = find_repo_root()
DOC_PATH = REPO_ROOT / "docs" / "data.md"
DATA_ROOT = REPO_ROOT / "data" / "futures"
BUND_FILES = {
    "roll_calendar": DATA_ROOT / "roll_calendars_csv" / "BUND.csv",
    "multiple_prices": DATA_ROOT / "multiple_prices_csv" / "BUND.csv",
    "adjusted_prices": DATA_ROOT / "adjusted_prices_csv" / "BUND.csv",
}
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)


In [ ]:
heading_pattern = re.compile(r"^(#{1,3})\s+(.*)$")
headings = []
for line in DOC_PATH.read_text(encoding="utf-8").splitlines():
    match = heading_pattern.match(line)
    if match:
        headings.append({"level": len(match.group(1)), "title": match.group(2)})

pd.DataFrame(headings).head(20)


## Workflow overview

Part 1 of `docs/data.md` describes a dependency chain: static instrument configuration and historical contract prices feed roll calendars; roll calendars plus contract prices feed multiple prices; multiple prices feed adjusted prices; adjusted prices, multiple prices, and FX data feed simulation and production layers.


In [ ]:
sorted(str(path.relative_to(REPO_ROOT)) for path in DATA_ROOT.iterdir())


## Inspecting shipped futures data

The main tutorial path stays inside the checkout. We start with the CSV configuration files and then inspect a single instrument across roll calendars, multiple prices, and adjusted prices.

The concrete files used here are `data/futures/csvconfig/instrumentconfig.csv`, `data/futures/csvconfig/rollconfig.csv`, `data/futures/csvconfig/spreadcosts.csv`, `data/futures/roll_calendars_csv/BUND.csv`, `data/futures/multiple_prices_csv/BUND.csv`, and `data/futures/adjusted_prices_csv/BUND.csv`.


In [ ]:
config_files = [
    DATA_ROOT / "csvconfig" / "instrumentconfig.csv",
    DATA_ROOT / "csvconfig" / "rollconfig.csv",
    DATA_ROOT / "csvconfig" / "spreadcosts.csv",
]

bund_examples = {
    label: pd.read_csv(path).head(5)
    for label, path in BUND_FILES.items()
}

{
    "config_paths": [str(path.relative_to(REPO_ROOT)) for path in config_files],
    "bund_files": [str(path.relative_to(REPO_ROOT)) for path in BUND_FILES.values()],
    "bund_examples": bund_examples,
}


In [ ]:
instrument_config = pd.read_csv(DATA_ROOT / "csvconfig" / "instrumentconfig.csv").head(5)
roll_config = pd.read_csv(DATA_ROOT / "csvconfig" / "rollconfig.csv").head(5)
spread_costs = pd.read_csv(DATA_ROOT / "csvconfig" / "spreadcosts.csv").head(5)

{
    "instrumentconfig.csv": instrument_config,
    "rollconfig.csv": roll_config,
    "spreadcosts.csv": spread_costs,
}


## Mapping document concepts to Python objects

The document's data layers map cleanly onto repo classes. The CSV readers expose the shipped files, and the higher-level sim objects compose those readers into something the rest of the system can consume.

The source files referenced in this walkthrough are `sysdata/data_blob.py`, `sysdata/csv/csv_futures_contract_prices.py`, `sysdata/csv/csv_roll_calendars.py`, `sysdata/csv/csv_multiple_prices.py`, `sysdata/csv/csv_adjusted_prices.py`, `sysdata/sim/csv_futures_sim_data.py`, and `sysdata/sim/db_futures_sim_data.py`.
This section stays runnable in a clean checkout because it reads those files as text instead of importing the optional-dependency-backed modules directly.


In [ ]:
SOURCE_FILES = {
    "csvFuturesContractPriceData": REPO_ROOT / "sysdata" / "csv" / "csv_futures_contract_prices.py",
    "csvRollCalendarData": REPO_ROOT / "sysdata" / "csv" / "csv_roll_calendars.py",
    "csvFuturesMultiplePricesData": REPO_ROOT / "sysdata" / "csv" / "csv_multiple_prices.py",
    "csvFuturesAdjustedPricesData": REPO_ROOT / "sysdata" / "csv" / "csv_adjusted_prices.py",
    "dataBlob": REPO_ROOT / "sysdata" / "data_blob.py",
    "csvFuturesSimData": REPO_ROOT / "sysdata" / "sim" / "csv_futures_sim_data.py",
    "dbFuturesSimData": REPO_ROOT / "sysdata" / "sim" / "db_futures_sim_data.py",
}


def class_declaration(path: Path, class_name: str) -> str:
    pattern = re.compile(rf"^class {re.escape(class_name)}\b.*$")
    for line in path.read_text(encoding="utf-8").splitlines():
        if pattern.match(line):
            return line.strip()
    return "<class declaration not found>"


object_map = pd.DataFrame(
    [
        {
            "object": name,
            "source_file": str(path.relative_to(REPO_ROOT)),
            "declaration": class_declaration(path, name),
        }
        for name, path in SOURCE_FILES.items()
    ]
)
object_map


## Interfaces and entry points

Part 4 of `docs/data.md` is where the storage objects become usable interfaces. `dataBlob` abstracts source-specific readers, `csvFuturesSimData` wires together the shipped CSV-backed data, and `dbFuturesSimData` shows the database-backed equivalent.

The object names above are the ones the smoke test checks for: `csvFuturesContractPriceData`, `csvRollCalendarData`, `csvFuturesMultiplePricesData`, `csvFuturesAdjustedPricesData`, `csvFuturesSimData`, `dbFuturesSimData`, and `dataBlob`.
The implementation details are pulled from the source files so this path stays safe even when optional broker or database dependencies are unavailable.


In [ ]:
interface_map = object_map.loc[
    object_map["object"].isin(["dataBlob", "csvFuturesSimData", "dbFuturesSimData"]),
    ["object", "source_file", "declaration"],
].rename(columns={"object": "interface"})
interface_map


## Optional integration examples

These examples are guarded on purpose. They show where the notebook connects to MongoDB-backed simulation data and Interactive Brokers without making the clean-checkout walkthrough depend on either service.


### MongoDB

If you have MongoDB and Parquet configured, you can switch this example on to build the database-backed futures simulation interface. The imports stay inside the guard so the notebook still runs in a fresh checkout with no database services available.


In [ ]:
RUN_MONGODB_EXAMPLE = False

if RUN_MONGODB_EXAMPLE:
    from sysdata.data_blob import dataBlob
    from sysdata.sim.db_futures_sim_data import dbFuturesSimData

    data = dataBlob()
    sim_data = dbFuturesSimData(data=data)
    print(sim_data)
else:
    print("Set RUN_MONGODB_EXAMPLE = True after configuring MongoDB and Parquet.")


### Interactive Brokers

If you have an IB Gateway or TWS session available, you can switch this on to seed contract prices from Interactive Brokers. The example remains guarded so the notebook does not try to open a broker connection during normal walkthrough use.


In [ ]:
RUN_INTERACTIVE_BROKERS_EXAMPLE = False

if RUN_INTERACTIVE_BROKERS_EXAMPLE:
    from sysinit.futures.seed_price_data_from_IB import seed_price_data_from_IB

    seed_price_data_from_IB("BUND")
else:
    print(
        "Set RUN_INTERACTIVE_BROKERS_EXAMPLE = True after configuring IB Gateway and private credentials."
    )


## Where to go next

If you want to keep going after this walkthrough, the most useful follow-on reads are `docs/backtesting.md` for simulation usage, `docs/production.md` for the production stack, and `docs/IB.md` for broker connectivity.

For the writable paths that feed these examples, the main scripts live under `sysinit/futures/`, especially the CSV-to-database loaders and the Interactive Brokers seeding script.
